<a href="https://colab.research.google.com/github/ahmed-chrif/PCD-Smart-Scarecrow/blob/main/audio_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this in a SEPARATE notebook while first notebook copies to Drive
import os, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# Mount Drive first
from google.colab import drive
drive.mount('/drive')

BACKGROUND_DIR = "/drive/MyDrive/dataset/background"  # write directly to Drive
os.makedirs(BACKGROUND_DIR, exist_ok=True)

def count_wavs():
    return len(list(Path(BACKGROUND_DIR).glob("*.wav")))

def process_chunk(args):
    src, i = args
    try:
        out_file = os.path.join(BACKGROUND_DIR, f"{src.stem}_chunk{i:04d}.wav")
        if os.path.exists(out_file):
            return 1
        subprocess.run([
            "ffmpeg", "-y", "-loglevel", "error",
            "-ss", str(i), "-t", "1",
            "-i", str(src),
            "-ac", "1", "-ar", "16000", "-sample_fmt", "s16",
            out_file
        ], check=True, timeout=15)
        return 1
    except:
        return 0

print(f"💾 Currently on Drive: {count_wavs():,} / 63,000")

# ── Download URBAN-SED ─────────────────────────────────────────
print("📥 Downloading URBAN-SED...")
os.system("wget -q --show-progress "
          "https://zenodo.org/record/1324404/files/URBAN-SED_v2.0.0.tar.gz "
          "-O /content/URBAN-SED.tar.gz")
os.system("tar -xzf /content/URBAN-SED.tar.gz -C /content/")
os.system("rm /content/URBAN-SED.tar.gz")

# Filter junk files
all_files = list(Path("/content/URBAN-SED_v2.0.0").rglob("*.wav"))
real_files = [f for f in all_files if not f.name.startswith("._")]
print(f"✅ {len(real_files):,} real files found")

# Build chunk tasks for exactly what we need
needed = max(0, 63000 - count_wavs())
print(f"➕ Need {needed:,} more chunks")

tasks = []
for f in real_files:
    for i in range(10):
        out = os.path.join(BACKGROUND_DIR, f"{f.stem}_chunk{i:04d}.wav")
        if not os.path.exists(out):
            tasks.append((f, i))
    if len(tasks) >= needed + 500:
        break

print(f"🎯 Queued {len(tasks):,} chunk tasks")

start = time.time()
total = 0
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(process_chunk, t): t for t in tasks}
    for done, future in enumerate(as_completed(futures), 1):
        total += future.result()
        if done % 500 == 0 or done == len(tasks):
            elapsed = time.time() - start
            rate = done / max(elapsed, 1)
            eta = (len(tasks) - done) / max(rate, 0.001)
            current = count_wavs()
            print(f"   ↳ {done}/{len(tasks)} | {current:,} on Drive | "
                  f"{rate:.1f} chunks/sec | ~{eta/60:.1f} min left")
        if count_wavs() >= 63000:
            print("🎉 Target reached!")
            break

print(f"\n{'='*50}")
final = count_wavs()
print(f"✅ Done!")
print(f"📊 Final on Drive : {final:,}")
print(f"➕ Still needed   : {max(0, 63000-final):,}")
print(f"{'='*50}")